# Through-Ball Playbook — Barcelona 2010–11

Turns broadcast footage of three Barça through-ball goals into coach-style breakdowns:
a tracked video per clip (ellipses under players, IDs, run trails, ball marker) plus
freeze-frame PNGs with the **pass arrow**, the **run arrow**, and a **shaded zone**
showing the space attacked. You record the voiceover on top.

## How to run
1. **Runtime > Change runtime type > T4 GPU** (free tier is fine).
2. Run **CELL 1** first, every session — it installs, mounts Drive, and defines helpers.
3. Then run cells top to bottom. Every output is written to Drive immediately, so a dead
   session loses nothing. After a restart: re-run CELL 1, then re-run any cell whose
   functions you need (they're cheap — the heavy work is skipped if its output exists).

**Everything lands in `MyDrive/soccer_analysis/`** — `raw/` (downloads), `clips/`
(trimmed sequences), `outputs/` (tracked MP4s, caches, playbook PNGs).

## The clips
| clip name | match | moment |
|---|---|---|
| `pedro_ucl_final` | Barcelona 3–1 Man Utd, UCL Final, 28 May 2011 | Pedro ~27' — Xavi's through ball, right channel |
| `pedro_ucl_semi` | Barcelona 1–1 Real Madrid, UCL SF 2nd leg, 3 May 2011 | Pedro ~54' — Iniesta releases Pedro in behind |
| `villa_clasico_55min` | Barcelona 5–0 Real Madrid, 29 Nov 2010 | Villa ~55' — ball in behind the back line |
| `villa_clasico_58min` | same match | Villa ~58' — same pattern again |

## Straight talk (read once)
- **Ball detection on broadcast footage is genuinely unreliable** — tiny, fast,
  motion-blurred. This notebook runs detection at 1280 px, accepts low-confidence ball
  hits, and interpolates gaps up to 1 s. CELL 5 prints the hit rate per clip; if it's
  under ~30%, mark the ball by hand on the key frames in CELL 7. The player tracking
  (which is what the arrows are built from) does **not** depend on the ball.
- **Pass detection is manual by design for v1**: you supply 2 frame numbers and 2 tracker
  IDs per clip (CELL 8). That's 4 numbers — more robust than any auto-detector here.
- Old footage means some missed detections in crowded shots. Fine — through-ball
  moments happen in open space.
- This is broadcast/UEFA footage: keep the published excerpts short and lead with your
  own analysis/voiceover (the freeze-frames + commentary are the transformative part).

*v1 scope: through-ball execution only. Phase 2 (build-up patterns, midfield rotations)
comes later.*


In [ ]:
# CELL 1 — SETUP. Run first, every session (~1 min).
!pip install -q "supervision==0.28.0" "trackers==2.4.0" "ultralytics>=8.2" yt-dlp

import os, pickle
import numpy as np
import cv2
import matplotlib.pyplot as plt
import supervision as sv
from trackers import ByteTrackTracker

from google.colab import drive
drive.mount("/content/drive")

BASE  = "/content/drive/MyDrive/soccer_analysis"
RAW   = f"{BASE}/raw"      # full downloaded highlight videos
CLIPS = f"{BASE}/clips"    # trimmed 10-20 s sequences
OUT   = f"{BASE}/outputs"  # tracked videos, caches, playbook PNGs
for d in (RAW, CLIPS, OUT):
    os.makedirs(d, exist_ok=True)

def show_image(img, title="", grid=False):
    plt.figure(figsize=(14, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    if grid:  # pixel grid every 100 px — for reading (x, y) off the image
        plt.grid(True, color="yellow", alpha=0.4)
        plt.xticks(range(0, img.shape[1], 100)); plt.yticks(range(0, img.shape[0], 100))
    else:
        plt.axis("off")
    plt.show()

def preview(video_path, start_s, end_s, n=8):
    """Grid of frames between start_s and end_s, timestamps + frame numbers under each."""
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    rows = int(np.ceil(n / 2))
    fig, axes = plt.subplots(rows, 2, figsize=(14, 4 * rows))
    for ax, t in zip(np.array(axes).flat, np.linspace(start_s, end_s, n)):
        cap.set(cv2.CAP_PROP_POS_MSEC, t * 1000)
        ok, frame = cap.read()
        if ok:
            ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            ax.set_title(f"t = {t:.1f} s   frame ~{int(t * fps)}", fontsize=11)
        ax.axis("off")
    plt.tight_layout(); plt.show()
    cap.release()

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU, then re-run this cell")
print("Everything saves under:", BASE)


## Step 1 — Get the footage

CELL 2 downloads the top YouTube search hit for each match into Drive.
CELL 3 lets you scrub a video as a grid of frames to find the through-ball moment.
CELL 4 cuts the clips.

If CELL 3 shows a download is the wrong video, delete that file from
`MyDrive/soccer_analysis/raw/` and replace its search query in CELL 2 with a direct
YouTube URL (just paste the URL in place of `ytsearch1:...`), then re-run CELL 2.


In [ ]:
# CELL 2 — DOWNLOAD the three highlight videos into Drive (skips ones already there).
MATCHES = {
    "ucl_final_2011": "ytsearch1:Barcelona Manchester United 3-1 2011 Champions League final highlights",
    "ucl_semi_2011":  "ytsearch1:Barcelona Real Madrid 1-1 2011 Champions League semi final second leg highlights Pedro",
    "clasico_5_0":    "ytsearch1:Barcelona Real Madrid 5-0 November 2010 La Liga highlights",
}

for name, query in MATCHES.items():
    path = f"{RAW}/{name}.mp4"
    if os.path.exists(path):
        print("already in Drive:", name)
        continue
    print("fetching:", name)
    !yt-dlp -f "mp4[height<=720]/best[height<=720]/best" --no-playlist --merge-output-format mp4 -o "{path}" "{query}"

!ls -lh {RAW}


In [ ]:
# CELL 3 — FIND the through-ball moment.
# Pick a video and a rough window, run, read the timestamps under the frames.
# Narrow WINDOW and re-run until you can see the whole sequence: pass -> run -> finish.
# Goals in highlight reels are usually in chronological order, so the ~27' Pedro goal
# sits early in the UCL final reel; Villa's 55'/58' goals sit past the middle of the 5-0 reel.
VIDEO  = f"{RAW}/ucl_final_2011.mp4"
WINDOW = (0, 180)   # (start, end) in seconds — start wide, then zoom in

preview(VIDEO, *WINDOW, n=10)


In [ ]:
# CELL 4 — TRIM. One line per clip: (source video, start "MM:SS", length in seconds).
# Start each clip ~5 s BEFORE the pass is released. Fill the starts in from CELL 3,
# run, and check the previews below — every frame of the sequence should be inside the cut.
TRIMS = {
    "pedro_ucl_final":     ("ucl_final_2011", "01:30", 18),   # <- placeholder starts,
    "pedro_ucl_semi":      ("ucl_semi_2011",  "01:00", 18),   #    set from CELL 3
    "villa_clasico_55min": ("clasico_5_0",    "04:00", 18),
    "villa_clasico_58min": ("clasico_5_0",    "05:00", 18),
}

for clip, (src, start, secs) in TRIMS.items():
    out_path = f"{CLIPS}/{clip}.mp4"
    # re-encode instead of "-c copy" so the cut is frame-accurate, audio dropped
    !ffmpeg -y -loglevel error -ss "00:{start}" -t {secs} -i "{RAW}/{src}.mp4" -c:v libx264 -preset fast -crf 20 -an "{out_path}"
    print("cut:", out_path)

for clip, (_, _, secs) in TRIMS.items():
    print("\n===", clip, "===")
    preview(f"{CLIPS}/{clip}.mp4", 0, secs - 0.2, n=6)


## Step 2 — Detect + track

CELL 5 runs YOLOv8x over one clip (players + ball) and saves a **position cache** to
Drive; CELL 6 renders the tracked video from that cache and prints the tracker-ID table.
Inference runs once per clip — everything after (re-renders, freeze-frames) reuses the
cache and takes seconds.

**Get one clip looking right before batching the rest (CELL 9).**


In [ ]:
# CELL 5 — DETECT + TRACK one clip (~2-3 min on a T4).
from ultralytics import YOLO

CLIP = "pedro_ucl_final"

model = YOLO("yolov8x.pt")

def fill_ball_gaps(ball_raw, fps, max_gap_s=1.0):
    """Linear-interpolate the ball across short detection gaps.
    Returns {frame: (x, y, was_interpolated)}."""
    filled, idxs = {}, sorted(ball_raw)
    for a, b in zip(idxs, idxs[1:]):
        filled[a] = (*ball_raw[a], False)
        if b - a <= max_gap_s * fps:
            for k in range(a + 1, b):
                t = (k - a) / (b - a)
                filled[k] = (ball_raw[a][0] + t * (ball_raw[b][0] - ball_raw[a][0]),
                             ball_raw[a][1] + t * (ball_raw[b][1] - ball_raw[a][1]), True)
    if idxs:
        filled[idxs[-1]] = (*ball_raw[idxs[-1]], False)
    return filled

def track_clip(clip_name):
    video = f"{CLIPS}/{clip_name}.mp4"
    info = sv.VideoInfo.from_video_path(video)
    tracker = ByteTrackTracker(frame_rate=info.fps,
                               lost_track_buffer=int(info.fps * 2),
                               track_activation_threshold=0.4,
                               minimum_consecutive_frames=2)
    frames, ball_raw = [], {}
    for i, frame in enumerate(sv.get_video_frames_generator(video)):
        det = sv.Detections.from_ultralytics(model(frame, imgsz=1280, conf=0.15, verbose=False)[0])
        players = tracker.update(det[(det.class_id == 0) & (det.confidence > 0.35)])
        frames.append({"xyxy": players.xyxy.copy(),
                       "ids": None if players.tracker_id is None else players.tracker_id.copy()})
        ball = det[det.class_id == 32]
        if len(ball):  # keep only the most confident ball hit per frame
            x1, y1, x2, y2 = ball.xyxy[int(np.argmax(ball.confidence))]
            ball_raw[i] = (float(x1 + x2) / 2, float(y1 + y2) / 2)
    cache = {"clip": clip_name, "video": video, "fps": info.fps,
             "wh": (info.width, info.height), "n_frames": len(frames),
             "frames": frames, "ball": fill_ball_gaps(ball_raw, info.fps)}
    with open(f"{OUT}/{clip_name}_cache.pkl", "wb") as f:
        pickle.dump(cache, f)
    pct = 100 * len(ball_raw) / max(len(frames), 1)
    print(f"{clip_name}: {len(frames)} frames at {info.fps:.0f} fps | "
          f"ball detected on {len(ball_raw)} frames ({pct:.0f}%)")
    if pct < 30:
        print("  -> ball tracking is weak on this clip. The arrows don't need it; "
              "if you want the ball dot on the freeze-frames, set it by hand in CELL 7.")
    return cache

cache = track_clip(CLIP)


In [ ]:
# CELL 6 — RENDER the tracked video + read off your tracker IDs.
def load_cache(clip_name):
    with open(f"{OUT}/{clip_name}_cache.pkl", "rb") as f:
        return pickle.load(f)

def detections_at(cache, i):
    fr = cache["frames"][i]
    if fr["ids"] is None or len(fr["xyxy"]) == 0:
        return sv.Detections.empty()
    n = len(fr["xyxy"])
    return sv.Detections(xyxy=fr["xyxy"], class_id=np.zeros(n, dtype=int),
                         confidence=np.ones(n), tracker_id=fr["ids"])

def draw_ball(img, cache, i):
    if i in cache["ball"]:
        x, y, interp = cache["ball"][i]
        # solid yellow dot when detected, hollow when interpolated
        cv2.circle(img, (int(x), int(y)), 7, (0, 215, 255), 2 if interp else -1, cv2.LINE_AA)
    return img

def render_tracked(clip_name):
    cache = load_cache(clip_name)
    ellipse = sv.EllipseAnnotator(color_lookup=sv.ColorLookup.TRACK, thickness=2)
    labels  = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.45,
                                text_padding=4, text_position=sv.Position.BOTTOM_CENTER)
    trace   = sv.TraceAnnotator(color_lookup=sv.ColorLookup.TRACK,
                                trace_length=int(cache["fps"] * 2.5), thickness=3)
    target = f"{OUT}/{clip_name}_tracked.mp4"
    info = sv.VideoInfo.from_video_path(cache["video"])
    with sv.VideoSink(target, info) as sink:
        for i, frame in enumerate(sv.get_video_frames_generator(cache["video"])):
            det, out = detections_at(cache, i), frame.copy()
            if len(det):
                out = ellipse.annotate(out, det)
                out = labels.annotate(out, det, labels=[f"#{t}" for t in det.tracker_id])
                out = trace.annotate(out, det)
            sink.write_frame(draw_ball(out, cache, i))
    print(f"saved: {target}")
    print(f"open it in the Drive app to scrub; frame number ~= seconds x {cache['fps']:.0f}")

def show_tracked_frame(clip_name, i, grid=False):
    """One annotated frame — use to confirm tracker IDs and frame numbers.
    grid=True overlays a 100 px grid for reading (x, y) coordinates."""
    cache = load_cache(clip_name)
    cap = cv2.VideoCapture(cache["video"]); cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, frame = cap.read(); cap.release()
    det, out = detections_at(cache, i), frame.copy()
    if len(det):
        out = sv.EllipseAnnotator(color_lookup=sv.ColorLookup.TRACK, thickness=2).annotate(out, det)
        out = sv.LabelAnnotator(color_lookup=sv.ColorLookup.TRACK, text_scale=0.45, text_padding=4,
                                text_position=sv.Position.BOTTOM_CENTER
                                ).annotate(out, det, labels=[f"#{t}" for t in det.tracker_id])
    show_image(draw_ball(out, cache, i), f"{clip_name} - frame {i}", grid=grid)

def id_summary(clip_name, top=15):
    """Longest-lived tracker IDs: the passer and receiver are usually near the top."""
    cache, seen = load_cache(clip_name), {}
    for i, fr in enumerate(cache["frames"]):
        if fr["ids"] is None:
            continue
        for t in fr["ids"]:
            s = seen.setdefault(int(t), [i, i, 0]); s[1] = i; s[2] += 1
    print(f"{'id':>5} {'first':>6} {'last':>6} {'frames':>7}")
    for t, (a, b, n) in sorted(seen.items(), key=lambda kv: -kv[1][2])[:top]:
        print(f" #{t:<4} {a:>6} {b:>6} {n:>7}")

render_tracked(CLIP)
id_summary(CLIP)
show_tracked_frame(CLIP, 60)


## Step 3 — Playbook freeze-frames

You supply **4 numbers per clip** in CELL 8:
- `release` — frame where the pass leaves the passer's foot
- `reception` — frame where the receiver touches it
- `passer`, `receiver` — their tracker IDs

Finding them from a phone: open `outputs/<clip>_tracked.mp4` in the Drive app, scrub to
the moment, multiply seconds by the fps that CELL 6 printed, then confirm with
`show_tracked_frame(CLIP, frame_number)`. The `id_summary` table helps — the two
protagonists are usually among the longest-lived IDs. If a tracker ID switches mid-clip
(it happens on occlusions), use the ID that is valid **at the frames you picked**.

CELL 7 is only for fixing the ball dot by hand — skip it unless CELL 5 told you otherwise.


In [ ]:
# CELL 7 (OPTIONAL) — manual ball positions, only if CELL 5 reported a weak ball rate.
# Get (x, y) with show_tracked_frame(CLIP, frame, grid=True) and add 3-4 key frames here;
# the gaps between them are interpolated. Then re-run render_tracked(CLIP).
MANUAL_BALL = {
    # "pedro_ucl_final": {120: (640, 360), 135: (870, 310), 150: (1050, 330)},
}

for clip_name, points in MANUAL_BALL.items():
    cache = load_cache(clip_name)
    raw = {i: (x, y) for i, (x, y, interp) in cache["ball"].items() if not interp}
    raw.update({i: (float(x), float(y)) for i, (x, y) in points.items()})
    cache["ball"] = fill_ball_gaps(raw, cache["fps"])
    with open(f"{OUT}/{clip_name}_cache.pkl", "wb") as f:
        pickle.dump(cache, f)
    print(f"{clip_name}: ball now set on {len(cache['ball'])} frames - re-run render_tracked to see it")


In [ ]:
# CELL 8 — PLAYBOOK FREEZE-FRAMES: pass arrow + run arrow + shaded space.
# Fill in the 4 numbers per clip (see Step 3 note above). Outputs 2 PNGs per clip.
KEYFRAMES = {
    "pedro_ucl_final": dict(release=120, reception=185, passer=4, receiver=7),  # <- placeholders
}

YELLOW, CYAN = (0, 215, 255), (255, 220, 0)  # BGR: pass = yellow, run = cyan

def feet(cache, tid, i, search=15):
    """Bottom-centre of a player's box at frame i (nearest hit within +/- search frames)."""
    for di in sorted(range(-search, search + 1), key=abs):
        j = i + di
        if 0 <= j < cache["n_frames"]:
            fr = cache["frames"][j]
            if fr["ids"] is not None and tid in fr["ids"]:
                x1, y1, x2, y2 = fr["xyxy"][list(fr["ids"]).index(tid)]
                return int((x1 + x2) / 2), int(y2)
    raise ValueError(f"tracker #{tid} not found near frame {i} - "
                     f"confirm with show_tracked_frame('{cache['clip']}', {i})")

def run_path(cache, tid, i0, i1, step=3):
    pts = []
    for j in range(i0, i1 + 1, step):
        fr = cache["frames"][j]
        if fr["ids"] is not None and tid in fr["ids"]:
            x1, y1, x2, y2 = fr["xyxy"][list(fr["ids"]).index(tid)]
            pts.append((int((x1 + x2) / 2), int(y2)))
    return pts

def fat_arrow(img, p1, p2, color, thickness=5, tip=0.08):
    cv2.arrowedLine(img, p1, p2, (20, 20, 20), thickness + 4, cv2.LINE_AA, tipLength=tip)
    cv2.arrowedLine(img, p1, p2, color, thickness, cv2.LINE_AA, tipLength=tip)

def shade_zone(img, p1, p2, half_width=70, color=(80, 220, 80), alpha=0.30):
    """Semi-transparent corridor from p1 to p2 — the space being attacked."""
    v = np.array(p2, float) - np.array(p1, float)
    n = np.array([-v[1], v[0]]); n = n / (np.linalg.norm(n) + 1e-6) * half_width
    poly = np.array([p1 + n, p2 + n, p2 - n, p1 - n], dtype=np.int32)
    overlay = img.copy()
    cv2.fillPoly(overlay, [poly], color)
    return cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

def label_text(img, text, p, color):
    cv2.putText(img, text, p, cv2.FONT_HERSHEY_SIMPLEX, 0.9, (20, 20, 20), 5, cv2.LINE_AA)
    cv2.putText(img, text, p, cv2.FONT_HERSHEY_SIMPLEX, 0.9, color, 2, cv2.LINE_AA)

def read_frame(video, i):
    cap = cv2.VideoCapture(video); cap.set(cv2.CAP_PROP_POS_FRAMES, i)
    ok, frame = cap.read(); cap.release()
    return frame

def mark_players(img, det, passer, receiver):
    """Grey ellipses for everyone, yellow for the passer, cyan for the receiver."""
    img = sv.EllipseAnnotator(color="#c8c8c8", thickness=2).annotate(img, det)
    if det.tracker_id is not None:
        img = sv.EllipseAnnotator(color="#ffd700", thickness=4).annotate(img, det[det.tracker_id == passer])
        img = sv.EllipseAnnotator(color="#00dcff", thickness=4).annotate(img, det[det.tracker_id == receiver])
    return img

def draw_playbook(clip_name, release, reception, passer, receiver):
    cache = load_cache(clip_name)
    passer_pt  = feet(cache, passer, release)
    recv_start = feet(cache, receiver, release)
    recv_end   = feet(cache, receiver, reception)

    # frame 1 - moment of release: zone, pass lane, planned run
    img = read_frame(cache["video"], release)
    img = shade_zone(img, recv_start, recv_end)
    img = mark_players(img, detections_at(cache, release), passer, receiver)
    path = run_path(cache, receiver, release, reception)
    if len(path) >= 2:
        cv2.polylines(img, [np.array(path[:-1], np.int32)], False, (20, 20, 20), 9, cv2.LINE_AA)
        cv2.polylines(img, [np.array(path[:-1], np.int32)], False, CYAN, 5, cv2.LINE_AA)
        fat_arrow(img, path[-2], path[-1], CYAN)
    else:
        fat_arrow(img, recv_start, recv_end, CYAN)
    fat_arrow(img, passer_pt, recv_end, YELLOW)
    label_text(img, f"PASS #{passer}", (passer_pt[0] - 70, passer_pt[1] + 45), YELLOW)
    label_text(img, f"RUN #{receiver}", (recv_start[0] - 55, recv_start[1] - 25), CYAN)
    img = draw_ball(img, cache, release)
    p1 = f"{OUT}/{clip_name}_playbook_release.png"
    cv2.imwrite(p1, img); show_image(img, f"{clip_name} - release (frame {release})")

    # frame 2 - moment of reception: the run that was made
    img2 = read_frame(cache["video"], reception)
    img2 = mark_players(img2, detections_at(cache, reception), passer, receiver)
    if len(path) >= 2:
        cv2.polylines(img2, [np.array(path, np.int32)], False, (20, 20, 20), 9, cv2.LINE_AA)
        cv2.polylines(img2, [np.array(path, np.int32)], False, CYAN, 5, cv2.LINE_AA)
    label_text(img2, "RECEIVED", (recv_end[0] - 70, recv_end[1] + 45), CYAN)
    img2 = draw_ball(img2, cache, reception)
    p2 = f"{OUT}/{clip_name}_playbook_reception.png"
    cv2.imwrite(p2, img2); show_image(img2, f"{clip_name} - reception (frame {reception})")
    print("saved:", p1, "\nsaved:", p2)

for clip_name, kf in KEYFRAMES.items():
    draw_playbook(clip_name, **kf)


## Step 4 — Batch the rest + export

CELL 9 runs detect/track/render for every clip in `TRIMS` that isn't done yet
(~10 min total for three more clips on a T4). Then add each clip's 4 numbers to
`KEYFRAMES` in CELL 8 and re-run it. CELL 10 lists everything that's in Drive.

**Voiceover:** open the Drive folder from your phone, pull the `_tracked.mp4` files and
playbook PNGs into CapCut (or similar), and record over them.


In [ ]:
# CELL 9 — BATCH all remaining clips (skips ones already rendered).
for clip_name in TRIMS:
    if os.path.exists(f"{OUT}/{clip_name}_tracked.mp4"):
        print("already done:", clip_name)
        continue
    track_clip(clip_name)
    render_tracked(clip_name)
    id_summary(clip_name)
    print()
print("now add each clip's release/reception frames + IDs to KEYFRAMES in CELL 8 and re-run it")


In [ ]:
# CELL 10 — EXPORT CHECK: everything that's in Drive, ready for voiceover.
total = 0
for root, _, files in os.walk(BASE):
    for fname in sorted(files):
        p = os.path.join(root, fname)
        mb = os.path.getsize(p) / 1e6; total += mb
        print(f"{mb:7.1f} MB  {os.path.relpath(p, BASE)}")
print(f"\n{total:7.1f} MB total in {BASE}")

done = {
    "tracked videos":   sorted(f for f in os.listdir(OUT) if f.endswith("_tracked.mp4")),
    "playbook frames":  sorted(f for f in os.listdir(OUT) if f.endswith(".png")),
}
print()
for k, v in done.items():
    print(f"{k}: {len(v)}")
    for f in v:
        print("   ", f)
